# Radiomic feature extraction (EGD)

Run in the **radiomics** environment (`environment-radiomics.yml`, Python 3.7 + PyRadiomics 3.1.0a2).
Cells 1-5 are a single-patient sanity check; the batch extraction is further below.


In [ ]:
# considerar que para ejecutar radiomics se necesita una version muy antigua de python 3.7, 
# por eso este archivo solo se ocupa para generar la data

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import nibabel as nib

import SimpleITK as sitk
import six

from radiomics import featureextractor, getTestCase

import tempfile
import cv2

In [ ]:
t1_image_path = r"..\..\original_datasets\dataset_EGD\MR_EGD-0011\1_T1\NIFTI\T1.nii.gz"
t2_image_path = r"..\..\original_datasets\dataset_EGD\MR_EGD-0011\3_T2\NIFTI\T2.nii.gz"
flair_image_path = r"..\..\original_datasets\dataset_EGD\MR_EGD-0011\4_FLAIR\NIFTI\FLAIR.nii.gz"
mask_path = r"..\..\original_datasets\dataset_EGD\MR_EGD-0011\5_MASK\NIFTI\MASK.nii.gz"
mask_path = r"..\..\original_datasets\dataset_EGD\MR_EGD-0011\5_MASK\NIFTI\PERIPHERY_MASK.nii.gz"

In [ ]:
def save_volume_to_temp_file(volume, affine):
    """
    Save a 3D volume to a temporary NIfTI file and return the file path.
    """
    with tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False) as temp_file:
        temp_path = temp_file.name
        nib.save(nib.Nifti1Image(volume, affine), temp_path)
    return temp_path

def extract_features(image_path, mask_path):
    """
    Extract radiomics features from a given image and mask.
    """
    extractor = featureextractor.RadiomicsFeatureExtractor()
    #extractor.enableAllImageTypes()
    extractor.enableAllFeatures()
    extractor.addProvenance(False)
    return extractor.execute(image_path, mask_path)

# Load the original T1, T2 and FLAIR images
flair_nii_image = nib.load(flair_image_path)
t1_nii_image = nib.load(t1_image_path)
t2_nii_image = nib.load(t2_image_path)
flair_image_data = flair_nii_image.get_fdata()
t1_image_data = t1_nii_image.get_fdata()
t2_image_data = t2_nii_image.get_fdata()


# Extract features from original and processed images
print("Extracting features from T1, T2 and FLAIR images...")
flair_features = extract_features(flair_image_path, mask_path)
t1_features = extract_features(t1_image_path, mask_path)
t2_features = extract_features(t2_image_path, mask_path)

# Combine features from T1 and T2 images
combined_features = {}
for key, value in t1_features.items():
    combined_features[f"t1_{key}"] = value
for key, value in t2_features.items():
    combined_features[f"t2_{key}"] = value
for key, value in flair_features.items():
    combined_features[f"flair_{key}"] = value

# Display combined features
for idx, (key, value) in enumerate(combined_features.items()):
    print(f"{idx}. {key}: {value}")

In [ ]:
import nibabel as nib
import numpy as np
import tempfile
import os
from radiomics import featureextractor

def save_volume_to_temp_file(volume, affine):
    """
    Save a 3D volume to a temporary NIfTI file and return the file path.
    """
    with tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False) as temp_file:
        temp_path = temp_file.name
        nib.save(nib.Nifti1Image(volume, affine), temp_path)
    return temp_path

def extract_features(image_path, mask_path):
    """
    Extract radiomics features from a given image and mask.
    """
    extractor = featureextractor.RadiomicsFeatureExtractor()
    extractor.enableAllFeatures()
    extractor.enableAllImageTypes()
    extractor.addProvenance(False)
    return extractor.execute(image_path, mask_path)

def process_images_with_mirrored_mask(image_paths, mask_path, mirror=True):
    """
    Extract features for original and mirrored masks for given images.
    """
    # Load the original mask
    mask_nii = nib.load(mask_path)
    mask_data = mask_nii.get_fdata()
    
    if mirror:
        affine = mask_nii.affine

        # Flip the mask along the first axis
        mirrored_mask = mask_data[::-1, :, :]

        # Save the flipped mask to a temporary file
        mirrored_mask_path = save_volume_to_temp_file(mirrored_mask, affine)

    try:
        # Extract features for original and mirrored masks
        combined_features = {}
        contrasts=['t1', 't2', 'flair']
        
        for idx, image_path in enumerate(image_paths):
            if mirror:
                image_label = contrasts[idx]
                # Original mask features
                original_features = extract_features(image_path, mask_path)
                for key, value in original_features.items():
                    combined_features[f"{image_label}_original_{key}"] = value

                # Mirrored mask features
                mirrored_features = extract_features(image_path, mirrored_mask_path)
                for key, value in mirrored_features.items():
                    combined_features[f"{image_label}_mirrored_{key}"] = value
            else:
                image_label = contrasts[idx]
                # Original mask features
                original_features = extract_features(image_path, mask_path)
                for key, value in original_features.items():
                    combined_features[f"{image_label}_{key}"] = value

    finally:
        # Delete the temporary mirrored mask file
        if mirror and os.path.exists(mirrored_mask_path):
            os.remove(mirrored_mask_path)

    return combined_features

# File paths
image_paths = [t1_image_path, t2_image_path, flair_image_path]

# Extract features
print("Extracting features with original and mirrored masks...")
features = process_images_with_mirrored_mask(image_paths, mask_path, mirror=False)

# Display combined features
for idx, (key, value) in enumerate(features.items()):
    print(f"{idx}. {key}: {value}")


In [ ]:
labels_df = pd.read_excel("../../tabular_data/Genetic_and_Histological_labels.xlsx")
valid_patients = labels_df['Subject'][(labels_df['IDH']!=-1)|(labels_df['1p19q']!=-1)].tolist()
valid_patients

In [ ]:
labels_df

In [ ]:
import os
import re

# Base path to the folder containing the files
base_path = r"..\..\original_datasets\dataset_EGD"

# Create a dictionary to store T1, T2, and MASK pairs
file_pairs = {}

# Regex patterns for T1, T2, and MASK files
t1_pattern = re.compile(r"dataset_EGD\\MR_(EGD-\d{4})\\1_T1\\NIFTI\\T1\.nii\.gz$")
t2_pattern = re.compile(r"dataset_EGD\\MR_(EGD-\d{4})\\3_T2\\NIFTI\\T2\.nii\.gz$")
flair_pattern = re.compile(r"dataset_EGD\\MR_(EGD-\d{4})\\4_FLAIR\\NIFTI\\FLAIR\.nii\.gz$")

mask_pattern = re.compile(r"dataset_EGD\\MR_(EGD-\d{4})\\5_MASK\\NIFTI\\MASK\.nii\.gz$")

# Walk through the directory structure
for root, dirs, files in os.walk(base_path):
    for file_name in files:
        
        full_path = os.path.join(root, file_name)
        # Check for T1 match
        t1_match = t1_pattern.search(full_path)
        if t1_match:
            patient_id = t1_match.group(1)  # Extract EGD-XXXX
            if patient_id in valid_patients:
                if patient_id not in file_pairs:
                    file_pairs[patient_id] = {}
                file_pairs[patient_id]['T1'] = full_path
            continue

        # Check for T2 match
        t2_match = t2_pattern.search(full_path)
        if t2_match:
            patient_id = t2_match.group(1)  # Extract EGD-XXXX
            if patient_id in valid_patients:
                if patient_id not in file_pairs:
                    file_pairs[patient_id] = {}
                file_pairs[patient_id]['T2'] = full_path
            continue
            
        # Check for FLAIR match
        flair_match = flair_pattern.search(full_path)
        if flair_match:
            patient_id = flair_match.group(1)  # Extract EGD-XXXX
            if patient_id in valid_patients:
                if patient_id not in file_pairs:
                    file_pairs[patient_id] = {}
                file_pairs[patient_id]['FLAIR'] = full_path
            continue

        # Check for MASK match
        mask_match = mask_pattern.search(full_path)
        if mask_match:
            patient_id = mask_match.group(1)  # Extract EGD-XXXX
            if patient_id in valid_patients:
                if patient_id not in file_pairs:
                    file_pairs[patient_id] = {}
                file_pairs[patient_id]['MASK'] = full_path
            continue

# Process only the pairs where T1, T2, and MASK are available
for patient_id, paths in file_pairs.items():
    if 'T1' in paths and 'T2' in paths and 'FLAIR' in paths and 'MASK' in paths:
        print(f"Processing pair: T1={paths['T1']}, T2={paths['T2']}, FLAIR={paths['FLAIR']}, MASK={paths['MASK']}")
        # Perform your processing here
    else:
        print(f"Missing pair for patient {patient_id}: {paths}")


In [ ]:
len(file_pairs.items())

## Batch extraction (run twice)

The cell below extracts PyRadiomics features (original image + all filters) for every patient
with a known IDH or 1p/19q label. Each patient produces three rows, one per contrast (T1, T2, FLAIR).
It was run twice:

1. **Tumour + mirrored ROI**: keep only the first `mask_pattern` (`MASK.nii.gz`), call
   `process_images_with_masks(file_pairs, mirror=True)` and save to `../../radiomics_features.csv`.
2. **Boundary shell**: use the second `mask_pattern` (`PERIPHERY_MASK.nii.gz`, generated by
   `matlab/periphery_masks_EGD.m`), call `process_images_with_masks(file_pairs, mirror=False)` and
   save to `../../border_radiomics_features_EGD.csv` (current configuration).

Both CSVs are then reshaped to one row per patient by `processEGDcsv.ipynb`.


In [ ]:
import os
import re
import nibabel as nib
import numpy as np
import tempfile
from radiomics import featureextractor
import pandas as pd

# Function to save a volume to a temporary file
def save_volume_to_temp_file(volume, affine):
    with tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False) as temp_file:
        temp_path = temp_file.name
        nib.save(nib.Nifti1Image(volume, affine), temp_path)
    return temp_path

# Function to extract features using pyradiomics
def extract_features(image_path, mask_path):
    extractor = featureextractor.RadiomicsFeatureExtractor()
    extractor.enableAllImageTypes()
    extractor.enableAllFeatures()
    extractor.addProvenance(False)
    return extractor.execute(image_path, mask_path)

# Function to process images and extract features
def process_images_with_masks(file_pairs, mirror=True):
    results = []
    for key, files in file_pairs.items():
        print(f'Processing: {key}')
        t1_path = files['T1']
        t2_path = files['T2']
        flair_path = files['FLAIR']
        mask_path = files['MASK']

        # Load the original mask and flip it
        mask_nii = nib.load(mask_path)
        mask_data = mask_nii.get_fdata()
        if mirror:
            affine = mask_nii.affine
            mirrored_mask = mask_data[::-1, :, :]
            mirrored_mask_path = save_volume_to_temp_file(mirrored_mask, affine)

        try:
            # Process T1, T2, and FLAIR images with original and mirrored masks
            contrasts = {'T1': t1_path, 'T2': t2_path, 'FLAIR': flair_path}
            for contrast, image_path in contrasts.items():
                # Extract features with the original mask
                original_features = extract_features(image_path, mask_path)
                original_features = {f"{contrast}_original_{k}": v for k, v in original_features.items()}
                # Combine results into a single record
                record = {'Patient': key}
                record.update(original_features)
                
                if mirror:

                    # Extract features with the mirrored mask
                    mirrored_features = extract_features(image_path, mirrored_mask_path)
                    mirrored_features = {f"{contrast}_mirrored_{k}": v for k, v in mirrored_features.items()}

                    record.update(mirrored_features)
                    
                results.append(record)
        finally:
            # Delete the temporary mirrored mask file
            if mirror and os.path.exists(mirrored_mask_path):
                os.remove(mirrored_mask_path)

    return results

# Base path and valid patients
base_path = r"..\..\original_datasets\dataset_EGD"

# Extract file paths for T1, T2, FLAIR, and MASK
t1_pattern = re.compile(r"dataset_EGD\\MR_(EGD-\d{4})\\1_T1\\NIFTI\\T1\.nii\.gz$")
t2_pattern = re.compile(r"dataset_EGD\\MR_(EGD-\d{4})\\3_T2\\NIFTI\\T2\.nii\.gz$")
flair_pattern = re.compile(r"dataset_EGD\\MR_(EGD-\d{4})\\4_FLAIR\\NIFTI\\FLAIR\.nii\.gz$")
mask_pattern = re.compile(r"dataset_EGD\\MR_(EGD-\d{4})\\5_MASK\\NIFTI\\MASK\.nii\.gz$")
mask_pattern = re.compile(r"dataset_EGD\\MR_(EGD-\d{4})\\5_MASK\\NIFTI\\PERIPHERY_MASK\.nii\.gz$")

for root, dirs, files in os.walk(base_path):
    for file_name in files:
        full_path = os.path.join(root, file_name)
        if t1_pattern.search(full_path):
            patient_id = t1_pattern.search(full_path).group(1)
            if patient_id in valid_patients:
                file_pairs.setdefault(patient_id, {})['T1'] = full_path
        elif t2_pattern.search(full_path):
            patient_id = t2_pattern.search(full_path).group(1)
            if patient_id in valid_patients:
                file_pairs.setdefault(patient_id, {})['T2'] = full_path
        elif flair_pattern.search(full_path):
            patient_id = flair_pattern.search(full_path).group(1)
            if patient_id in valid_patients:
                file_pairs.setdefault(patient_id, {})['FLAIR'] = full_path
        elif mask_pattern.search(full_path):
            patient_id = mask_pattern.search(full_path).group(1)
            if patient_id in valid_patients:
                file_pairs.setdefault(patient_id, {})['MASK'] = full_path

# Filter file pairs to ensure all required paths are present
file_pairs = {k: v for k, v in file_pairs.items() if len(v) == 4}

# Process images and extract features
print("Extracting features from all images...")
features = process_images_with_masks(file_pairs, mirror=False)

# Save features to a CSV file
output_csv = r"..\..\border_radiomics_features_EGD.csv"
df = pd.DataFrame(features)
df.to_csv(output_csv, index=False)

print(f"Features extracted and saved to {output_csv}")


In [ ]:
import os
import re

# Base path to the folder containing the files
base_path = r"..\..\original_datasets\dataset_EGD"

# Create a dictionary to store FLAIR and MASK pairs
file_pairs = {}

# Regex patterns for T1, T2, and MASK files
flair_pattern = re.compile(r"dataset_EGD\\MR_(EGD-\d{4})\\4_FLAIR\\NIFTI\\FLAIR\.nii\.gz$")
mask_pattern = re.compile(r"dataset_EGD\\MR_(EGD-\d{4})\\5_MASK\\NIFTI\\MASK\.nii\.gz$")

# Walk through the directory structure
for root, dirs, files in os.walk(base_path):
    for file_name in files:
        
        full_path = os.path.join(root, file_name)
        # Check for T1 match
        flair_match = flair_pattern.search(full_path)
        if flair_match:
            patient_id = flair_match.group(1)  # Extract EGD-XXXX
            if patient_id in valid_patients:
                if patient_id not in file_pairs:
                    file_pairs[patient_id] = {}
                file_pairs[patient_id]['FLAIR'] = full_path
            continue


        # Check for MASK match
        mask_match = mask_pattern.search(full_path)
        if mask_match:
            patient_id = mask_match.group(1)  # Extract EGD-XXXX
            if patient_id in valid_patients:
                if patient_id not in file_pairs:
                    file_pairs[patient_id] = {}
                file_pairs[patient_id]['MASK'] = full_path
            continue

# Process only the pairs where FLAIR and MASK are available
for patient_id, paths in file_pairs.items():
    if 'FLAIR' in paths and 'MASK' in paths:
        print(f"Processing pair: FLAIR={paths['FLAIR']}, MASK={paths['MASK']}")
        # Perform your processing here
    else:
        print(f"Missing pair for patient {patient_id}: {paths}")


In [ ]:
# Constants to control feature extraction
EXTRACT_ORIGINAL = True  # Set to False to skip original features
EXTRACT_CLAHE = True     # Set to False to skip CLAHE features

# Initialize the feature extractor with custom settings
extractor = featureextractor.RadiomicsFeatureExtractor()
#extractor.enableAllImageTypes()
extractor.enableAllFeatures()

# Add preprocessing settings
extractor.settings['normalize'] = True
extractor.settings['normalizeScale'] = 100
extractor.settings['resampledPixelSpacing'] = [1, 1, 1]
extractor.settings['interpolator'] = 'sitkBSpline'
extractor.settings['binWidth'] = 25

# Optional: disable provenance tracking
extractor.addProvenance(False)

# Function to apply CLAHE to a 3D volume
def apply_clahe(volume):
    processed_slices = []
    for i in range(volume.shape[2]):
        slice_2d = volume[:, :, i]
        normalized_slice = cv2.normalize(slice_2d, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        processed_slice = clahe.apply(normalized_slice)
        processed_slices.append(processed_slice)
    return np.stack(processed_slices, axis=-1)

# Data to hold results
data = []

# Process each file pair
for key, files in file_pairs.items():
    print(f'Processing: {key}')
    flair_path = files['FLAIR']
    mask_path = files['MASK']
    
    # Load the FLAIR image
    flair_image = nib.load(flair_path)
    flair_data = flair_image.get_fdata()
    
    # Extract features for the original FLAIR image
    if EXTRACT_ORIGINAL:
        print(f'Extracting features for original FLAIR image: {flair_path}')
        flair_original_features = extractor.execute(flair_path, mask_path)
        flair_original_features = {f'flair_original_{k}': v for k, v in flair_original_features.items()}
    
    # Extract features for the CLAHE-processed T1 and T2 images
    if EXTRACT_CLAHE:
        print(f'Applying CLAHE and extracting features for FLAIR: {flair_path}')
        flair_processed_data = apply_clahe(flair_data)
        with tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False) as temp_file:
            flair_temp_path = temp_file.name
            flair_processed_nii = nib.Nifti1Image(flair_processed_data, affine=flair_image.affine)
            nib.save(flair_processed_nii, flair_temp_path)
        
        flair_clahe_features = extractor.execute(flair_temp_path, mask_path)
        flair_clahe_features = {f'flair_clahe_{k}': v for k, v in flair_clahe_features.items()}
        os.remove(flair_temp_path)
        
    
    # Combine results into a single row
    row = {'Image': key}
    if EXTRACT_ORIGINAL:
        row.update(flair_original_features)
    if EXTRACT_CLAHE:
        row.update(flair_clahe_features)
    
    data.append(row)

# Convert the list of dictionaries into a Pandas DataFrame
df = pd.DataFrame(data)

# Save to a CSV file
output_csv = r"..\..\radiomics_features_flair_EGD.csv"
df.to_csv(output_csv, index=False)

print(f"Features extracted and saved to {output_csv}")